# 📊 Multimodal RAG AI Teaching Assistant (統計迴歸分析 AI 助教)

本專案實作了一個基於多模態檢索增強生成 (Multimodal RAG) 的 AI 系統，專為解決大學統計學/迴歸分析課程中，全英教材閱讀門檻高、傳統 LLM 難以處理複雜 LaTeX 數學公式等痛點而設計。

* **使用模型**: Grok Vision Beta (OCR 視覺文字提取)、Grok 3 Mini Beta (生成回答)
* **檢索系統**: FAISS (本地向量資料庫) + `all-MiniLM-L6-v2` (Embedding)
* **介面**: Gradio
* **核心功能**: 支援 PDF/圖片上傳、精準解析講義內容、支援 LaTeX 數學公式前端渲染。

## 1. 環境建置與載入知識庫 (Environment Setup & Data Loading)

首先安裝專案所需的依賴套件，包含向量搜尋引擎 `faiss-cpu`、嵌入模型套件 `sentence-transformers`，以及處理非結構化資料與前端介面的 `pdf2image` 和 `gradio`。
為了加速執行效率，系統會自動從雲端下載並解壓縮預先建置好的「迴歸分析講義向量資料庫」(`lecture_embeddings.zip`)，避免每次執行都需重新切塊與計算向量。



In [ ]:
# 安裝必要的套件
!pip install faiss-cpu sentence-transformers requests pdf2image gdown gradio
!apt-get install poppler-utils

# 下載 lecture_embeddings.zip 從 Google Drive
import gdown
import os
import shutil

# Google Drive 直接下載連結
url = "https://drive.google.com/uc?export=download&id=1XWHtSlIER200sIDMd0P3vhFyWQuZ6LZe"
output = "lecture_embeddings.zip"

# 下載檔案
print("正在下載 lecture_embeddings.zip...")
gdown.download(url, output, quiet=False)

# 避免檔案衝突：如果 lecture_embeddings 資料夾存在，先刪除
if os.path.exists("lecture_embeddings"):
    shutil.rmtree("lecture_embeddings")

# 解壓縮 ZIP 檔案
print("正在解壓縮 lecture_embeddings.zip...")
!unzip lecture_embeddings.zip -d lecture_embeddings

## 2. 初始化向量索引與 Embedding 模型 (Vector DB Initialization)

將下載好的文字切塊 (Chunks) 與向量 (Embeddings) 載入記憶體，並使用 L2 距離建構 FAISS 索引 (`IndexFlatL2`)，以利後續進行高效的相似度檢索。同時載入輕量級的嵌入模型 `all-MiniLM-L6-v2`。

> **⚠️ 注意：** 為了確保資安，API Key 不應硬編碼 (Hardcode) 於腳本中。執行此區塊前，請確保已在 Colab 左側的 Secrets 面板中設定名為 `Grok` 的環境變數。

In [ ]:
# 載入向量和元數據
import numpy as np
import json
import faiss
from sentence_transformers import SentenceTransformer

embeddings = np.load("lecture_embeddings/embeddings.npy")
with open("lecture_embeddings/metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

# 建立 FAISS 索引
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print(f"已載入 {index.ntotal} 個向量")

# 載入嵌入模型
model = SentenceTransformer('all-MiniLM-L6-v2')

# 設定 API 密鑰（使用者需自行提供）
from google.colab import userdata
GROK_API_KEY = userdata.get('Grok').strip()  # 使用者需要在 Colab Secrets 中設置 "Grok" 密鑰

## 3. 多模態視覺解析模組 (Multimodal Vision Parsing)

本區塊設計 `process_file` 函數，專門處理使用者上傳的講義截圖或 PDF 檔案。
* **檔案預處理**：若為 PDF 檔案，系統會動態將其轉換為圖片格式，並進行 Base64 編碼。
* **高精度 OCR 提取**：呼叫 `Grok-vision-beta` 視覺模型，透過 Prompt Engineering 限制模型在提取文字時**必須保留原始的 LaTeX 數學公式格式**，解決了傳統 OCR 遇到統計學符號容易產生亂碼的痛點。

In [ ]:
# 處理圖片或 PDF 的函數
import base64
import requests
from pdf2image import convert_from_path

def process_file(file_path):
    if file_path is None:
        return None
    # 檢查檔案類型
    if file_path.endswith(".pdf"):
        # 將 PDF 轉成圖片
        images = convert_from_path(file_path, dpi=200)
        image_path = "temp_image.png"
        images[0].save(image_path, "PNG")  # 只處理第一頁
    else:
        image_path = file_path  # 假設是圖片（PNG/JPG）

    # 將圖片轉成 Base64
    with open(image_path, "rb") as image_file:
        image_base64 = base64.b64encode(image_file.read()).decode("utf-8")

    # 用 grok-vision-beta 提取文字
    headers = {
        "Authorization": f"Bearer {GROK_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "grok-vision-beta",
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "請提取圖片中的所有文字，並保留數學公式格式（LaTeX）。"
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{image_base64}"
                        }
                    }
                ]
            }
        ]
    }

    response = requests.post("https://api.x.ai/v1/chat/completions", headers=headers, json=payload)

    if response.status_code == 200:
        result = response.json()
        extracted_text = result["choices"][0]["message"]["content"]
        return extracted_text
    else:
        raise Exception(f"API 請求失敗：{response.status_code}, {response.text}")

## 4. RAG 檢索生成與 Web 介面部署 (RAG Engine & UI Deployment)

本區塊實作了 RAG 系統的核心大腦與使用者互動介面：
1. **相似度檢索 (Retrieval)**：將使用者的問題轉為向量，透過 FAISS 找出 Top-3 最相關的講義段落做為上下文 (Context)。
2. **增強生成 (Generation)**：將「檢索到的講義內容」與「圖片擷取出的文字」合併，餵給 `Grok-3-mini-beta` 進行精準回答，有效降低 LLM 面對專業統計問題時的幻覺 (Hallucination)。
3. **UI 渲染優化**：利用 `Gradio` 搭建網頁介面，並在底層透過字串替換，將 `\[ ... \]` 強制轉換為 `$$...$$`，確保前端 Markdown 引擎能完美渲染複雜的數學公式。

In [ ]:
# RAG 檢索與生成函數
def ask_question(query, extracted_text=None):
    # 將問題或提取的文字轉成向量
    if extracted_text:
        text_to_embed = extracted_text  # 如果有檔案，用提取的文字檢索
    else:
        text_to_embed = query  # 如果是純文字問題，用問題檢索

    query_embedding = model.encode([text_to_embed], convert_to_numpy=True)

    # 檢索最相關的 3 個片段
    k = 3
    distances, indices = index.search(query_embedding, k)

    # 獲取相關片段
    retrieved_chunks = [metadata[idx]["text"] for idx in indices[0]]
    retrieved_output = "檢索到的講義片段：\n"
    for i, chunk in enumerate(retrieved_chunks):
        retrieved_output += f"片段 {i+1}: {chunk[:200]}...\n"

    # 用 grok-3-mini-beta 生成答案
    headers = {
        "Authorization": f"Bearer {GROK_API_KEY}",
        "Content-Type": "application/json"
    }

    context = ""
    if extracted_text:
        context += f"檔案內容：\n{extracted_text}\n\n"
    context += "講義相關片段：\n" + "\n\n".join(retrieved_chunks)
    prompt = f"根據以下內容回答問題，保留數學公式格式（LaTeX）並用繁體中文回答，回答請以 Markdown 格式輸出，分步驟說明，公式使用 $$...$$ 包住以確保正確渲染：\n\n{context}\n\n問題：{query}"

    payload = {
        "model": "grok-3-mini-beta",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }

    response = requests.post("https://api.x.ai/v1/chat/completions", headers=headers, json=payload)

    if response.status_code == 200:
        result = response.json()
        answer = result["choices"][0]["message"]["content"]
        return retrieved_output, answer
    else:
        return retrieved_output, f"API 請求失敗：{response.status_code}, {response.text}"